# DistilBERT (starter) — Jigsaw toxic comments

Same layout as `03_bert.ipynb` but **`preprocess_for_distilbert`** and **DistilBERT** weights (fewer parameters, faster).

**Quick iteration:** In the next cell, set `QUICK_ITERATION = True` for a small subset (same idea as `03_bert.ipynb`); `False` uses the full train/validation split.

**Requires:** `pip install torch transformers`

In [ ]:
from pathlib import Path
import sys

# Repo root (contains preprocessing/)
ROOT = Path.cwd().resolve()
for c in (ROOT, ROOT.parent):
    if (c / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = c
        break
sys.path.insert(0, str(ROOT))
# notebooks/ for metrics_helpers
sys.path.insert(0, str(ROOT / "notebooks"))

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from preprocessing.text_preprocessing import LABEL_COLUMNS, preprocess_for_distilbert
from metrics_helpers import multilabel_evaluation_report, torch_parameter_count


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def binary_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    denom = 2 * tp + fp + fn
    return float((2 * tp) / denom) if denom > 0 else 0.0


def tune_per_label_thresholds(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    labels: list[str] | tuple[str, ...],
    threshold_grid: np.ndarray,
) -> tuple[dict[str, float], pd.DataFrame]:
    best_thresholds: dict[str, float] = {}
    threshold_rows: list[dict[str, float | str]] = []

    for j, label in enumerate(labels):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]

        best_t = 0.5
        best_f1 = -1.0
        for t in threshold_grid:
            y_pred_j = (y_prob_j >= t).astype(np.int64)
            f1_j = binary_f1(y_true_j, y_pred_j)
            if f1_j > best_f1:
                best_f1 = f1_j
                best_t = float(t)

        best_thresholds[label] = best_t
        threshold_rows.append(
            {
                "label": label,
                "best_threshold": round(best_t, 3),
                "best_f1_on_val": best_f1,
            }
        )

    threshold_df = pd.DataFrame(threshold_rows)
    return best_thresholds, threshold_df


def apply_thresholds(y_prob: np.ndarray, labels: list[str] | tuple[str, ...], thresholds: dict[str, float]) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=np.int64)
    for j, label in enumerate(labels):
        out[:, j] = (y_prob[:, j] >= thresholds[label]).astype(np.int64)
    return out


DEVICE = pick_device()
print("Device:", DEVICE)
torch.manual_seed(42)

QUICK_ITERATION = False

if QUICK_ITERATION:
    MAX_LENGTH = 64
    BATCH_SIZE = 16
    _train_n, _val_n = 512, 256
else:
    MAX_LENGTH = 128
    BATCH_SIZE = 8
    _train_n, _val_n = None, None

EPOCHS = 3
LR = 2e-5
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)

data = preprocess_for_distilbert(
    validation_fraction=0.1,
    random_state=42,
    max_length=MAX_LENGTH,
    return_tensors="pt",
    max_train_samples=_train_n,
    max_val_samples=_val_n,
)


def enc_dict(enc):
    keys = [k for k in ("input_ids", "attention_mask") if k in enc]
    return {k: enc[k] for k in keys}


train_enc = enc_dict(data.train_encodings)
val_enc = enc_dict(data.val_encodings)
y_train = data.y_train
y_val = data.y_val
y_val_np = np.asarray(y_val, dtype=np.int64)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(LABEL_COLUMNS),
    problem_type="multi_label_classification",
).to(DEVICE)


class EncDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


def collate(batch):
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0] if k != "labels"}
    out["labels"] = torch.stack([b["labels"] for b in batch], dim=0)
    return out


train_loader = DataLoader(EncDataset(train_enc, y_train), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(EncDataset(val_enc, y_val), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

opt = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn = torch.nn.BCEWithLogitsLoss()
num_params = torch_parameter_count(model)

print("CONFIG")
print("  mode:", "quick" if QUICK_ITERATION else "full-data")
print("  device:", DEVICE)
print("  train_samples:", len(y_train), "| val_samples:", len(y_val))
print("  max_length:", MAX_LENGTH, "| batch_size:", BATCH_SIZE)
print("  epochs (parity checkpoints):", EPOCHS)
print("  lr:", LR)
print("  num_parameters:", num_params)

records: list[dict[str, float | int]] = []
per_label_records: list[pd.DataFrame] = []
threshold_records: list[pd.DataFrame] = []

start_total = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss_sum = 0.0
    epoch_batches = 0
    epoch_steps = 0

    epoch_train_start = time.perf_counter()
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop("labels")

        opt.zero_grad()
        out = model(**batch)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        opt.step()

        epoch_loss_sum += float(loss.item())
        epoch_batches += 1
        epoch_steps += 1

    train_time_s = time.perf_counter() - epoch_train_start
    avg_train_loss = epoch_loss_sum / max(epoch_batches, 1)

    model.eval()
    all_probs: list[np.ndarray] = []
    val_loss_sum = 0.0
    val_batches = 0

    inf_start = time.perf_counter()
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop("labels")
            logits = model(**batch).logits
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())

            val_loss_sum += float(loss_fn(logits, labels).item())
            val_batches += 1
    inference_time_s = time.perf_counter() - inf_start

    prob_val = np.concatenate(all_probs, axis=0)
    pred_val_05 = (prob_val >= 0.5).astype(np.int64)

    per_label_05_df, summary_05 = multilabel_evaluation_report(y_val_np, pred_val_05, prob_val, LABEL_COLUMNS)

    best_thresholds, threshold_df = tune_per_label_thresholds(y_val_np, prob_val, LABEL_COLUMNS, THRESHOLD_GRID)
    pred_val_tuned = apply_thresholds(prob_val, LABEL_COLUMNS, best_thresholds)
    per_label_tuned_df, summary_tuned = multilabel_evaluation_report(y_val_np, pred_val_tuned, prob_val, LABEL_COLUMNS)

    per_label_compare_df = per_label_05_df.rename(
        columns={
            "precision": "precision_baseline",
            "recall": "recall_baseline",
            "f1": "f1_baseline",
            "roc_auc": "roc_auc_baseline",
        }
    ).merge(
        per_label_tuned_df.rename(
            columns={
                "precision": "precision_tuned",
                "recall": "recall_tuned",
                "f1": "f1_tuned",
                "roc_auc": "roc_auc_tuned",
            }
        ),
        on="label",
        how="left",
    )

    per_label_compare_df["epoch"] = epoch
    threshold_df["epoch"] = epoch

    records.append(
        {
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_loss": val_loss_sum / max(val_batches, 1),
            "train_time_s": train_time_s,
            "inference_time_s": inference_time_s,
            "train_time_cumulative_s": time.perf_counter() - start_total,
            "optimizer_steps": epoch_steps,
            "num_parameters": num_params,
            "f1_micro_baseline": summary_05["f1_micro"],
            "f1_macro_baseline": summary_05["f1_macro"],
            "f1_micro_tuned": summary_tuned["f1_micro"],
            "f1_macro_tuned": summary_tuned["f1_macro"],
        }
    )
    per_label_records.append(per_label_compare_df)
    threshold_records.append(threshold_df)

    print(
        f"Epoch {epoch}/{EPOCHS} | train_loss={avg_train_loss:.4f} | "
        f"val_loss={val_loss_sum / max(val_batches, 1):.4f} | "
        f"train_s={train_time_s:.1f} | infer_s={inference_time_s:.1f}"
    )
    print(
        f"  baseline micro/macro F1: {summary_05['f1_micro']:.4f} / {summary_05['f1_macro']:.4f}"
    )
    print(
        f"  tuned    micro/macro F1: {summary_tuned['f1_micro']:.4f} / {summary_tuned['f1_macro']:.4f}"
    )

epoch_summary_df = pd.DataFrame(records)
all_thresholds_df = pd.concat(threshold_records, ignore_index=True)
all_per_label_df = pd.concat(per_label_records, ignore_index=True)

print("\nEpoch-level benchmark summary:")
display(epoch_summary_df)

In [ ]:
epoch_metrics_df = epoch_summary_df.copy()
epoch_metrics_df["f1_micro_delta_tuned_minus_baseline"] = (
    epoch_metrics_df["f1_micro_tuned"] - epoch_metrics_df["f1_micro_baseline"]
)
epoch_metrics_df["f1_macro_delta_tuned_minus_baseline"] = (
    epoch_metrics_df["f1_macro_tuned"] - epoch_metrics_df["f1_macro_baseline"]
)

print("Epoch comparison table (baseline and tuned):")
display(
    epoch_metrics_df[
        [
            "epoch",
            "train_loss",
            "val_loss",
            "train_time_s",
            "inference_time_s",
            "num_parameters",
            "f1_micro_baseline",
            "f1_macro_baseline",
            "f1_micro_tuned",
            "f1_macro_tuned",
            "f1_micro_delta_tuned_minus_baseline",
            "f1_macro_delta_tuned_minus_baseline",
        ]
    ]
)

final_epoch = int(epoch_metrics_df["epoch"].max())
final_thresholds_df = (
    all_thresholds_df.loc[all_thresholds_df["epoch"] == final_epoch]
    .sort_values("label")
    .reset_index(drop=True)
)
final_per_label_df = (
    all_per_label_df.loc[all_per_label_df["epoch"] == final_epoch]
    .sort_values("label")
    .reset_index(drop=True)
)

print(f"\nFinal-epoch ({final_epoch}) threshold table:")
display(final_thresholds_df)

print(f"Final-epoch ({final_epoch}) per-label baseline vs tuned metrics:")
display(
    final_per_label_df[
        [
            "label",
            "precision_baseline",
            "recall_baseline",
            "f1_baseline",
            "roc_auc_baseline",
            "precision_tuned",
            "recall_tuned",
            "f1_tuned",
            "roc_auc_tuned",
        ]
    ]
)

In [ ]:
import matplotlib.pyplot as plt

# 1) Train/validation loss vs epoch
plt.figure(figsize=(7, 4))
plt.plot(epoch_metrics_df["epoch"], epoch_metrics_df["train_loss"], marker="o", label="train_loss")
plt.plot(epoch_metrics_df["epoch"], epoch_metrics_df["val_loss"], marker="o", label="val_loss")
plt.title("DistilBERT Full-Data: Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("BCEWithLogits loss")
plt.xticks(epoch_metrics_df["epoch"])
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 2) Micro/Macro F1 vs epoch (baseline and tuned)
plt.figure(figsize=(8, 4.5))
plt.plot(epoch_metrics_df["epoch"], epoch_metrics_df["f1_micro_baseline"], marker="o", label="micro F1 (0.5)")
plt.plot(epoch_metrics_df["epoch"], epoch_metrics_df["f1_macro_baseline"], marker="o", label="macro F1 (0.5)")
plt.plot(epoch_metrics_df["epoch"], epoch_metrics_df["f1_micro_tuned"], marker="o", linestyle="--", label="micro F1 (tuned)")
plt.plot(epoch_metrics_df["epoch"], epoch_metrics_df["f1_macro_tuned"], marker="o", linestyle="--", label="macro F1 (tuned)")
plt.title("DistilBERT Full-Data: F1 vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.xticks(epoch_metrics_df["epoch"])
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 3) Train time and inference time vs epoch
plt.figure(figsize=(7, 4))
plt.plot(epoch_metrics_df["epoch"], epoch_metrics_df["train_time_s"], marker="o", label="train_time_s")
plt.plot(epoch_metrics_df["epoch"], epoch_metrics_df["inference_time_s"], marker="o", label="inference_time_s")
plt.title("DistilBERT Full-Data: Time vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Seconds")
plt.xticks(epoch_metrics_df["epoch"])
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 4) Baseline vs tuned delta plot across epochs
plt.figure(figsize=(7, 4))
plt.plot(
    epoch_metrics_df["epoch"],
    epoch_metrics_df["f1_micro_delta_tuned_minus_baseline"],
    marker="o",
    label="delta micro F1 (tuned-baseline)",
)
plt.plot(
    epoch_metrics_df["epoch"],
    epoch_metrics_df["f1_macro_delta_tuned_minus_baseline"],
    marker="o",
    label="delta macro F1 (tuned-baseline)",
)
plt.axhline(0.0, color="black", linewidth=1, linestyle="--")
plt.title("Threshold Tuning Gain vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("F1 delta")
plt.xticks(epoch_metrics_df["epoch"])
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 5) Final-epoch per-label precision/recall/F1 bars
metrics_to_plot = [
    ("precision_baseline", "precision_tuned", "Precision"),
    ("recall_baseline", "recall_tuned", "Recall"),
    ("f1_baseline", "f1_tuned", "F1"),
]

x = np.arange(len(final_per_label_df["label"]))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (base_col, tuned_col, label_name) in zip(axes, metrics_to_plot):
    ax.bar(x - width / 2, final_per_label_df[base_col], width=width, label="baseline_0.5")
    ax.bar(x + width / 2, final_per_label_df[tuned_col], width=width, label="tuned_threshold")
    ax.set_title(f"Final Epoch: {label_name}")
    ax.set_xticks(x)
    ax.set_xticklabels(final_per_label_df["label"], rotation=30, ha="right")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Score")
axes[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

best_epoch_idx = epoch_metrics_df["f1_macro_tuned"].idxmax()
best_epoch_row = epoch_metrics_df.loc[best_epoch_idx]

print("--- DistilBERT epoch-parity benchmark summary ---")
print(f"  mode: {'quick' if QUICK_ITERATION else 'full-data'}")
print(f"  epochs logged: {EPOCHS}")
print(f"  device: {DEVICE}")
print(f"  num_parameters: {int(best_epoch_row['num_parameters'])}")
print(
    f"  final_epoch={final_epoch}: "
    f"baseline micro/macro F1 = {epoch_metrics_df.loc[epoch_metrics_df['epoch'] == final_epoch, 'f1_micro_baseline'].iloc[0]:.4f} / "
    f"{epoch_metrics_df.loc[epoch_metrics_df['epoch'] == final_epoch, 'f1_macro_baseline'].iloc[0]:.4f}"
)
print(
    f"  final_epoch={final_epoch}: "
    f"tuned micro/macro F1 = {epoch_metrics_df.loc[epoch_metrics_df['epoch'] == final_epoch, 'f1_micro_tuned'].iloc[0]:.4f} / "
    f"{epoch_metrics_df.loc[epoch_metrics_df['epoch'] == final_epoch, 'f1_macro_tuned'].iloc[0]:.4f}"
)
print(
    f"  best_macro_epoch={int(best_epoch_row['epoch'])}: "
    f"macro F1 (tuned) = {best_epoch_row['f1_macro_tuned']:.4f}"
)
if (epoch_metrics_df["f1_macro_delta_tuned_minus_baseline"] >= 0).all():
    print("  threshold tuning improved or matched macro-F1 at every epoch checkpoint.")
else:
    print("  threshold tuning did not improve macro-F1 at every checkpoint; inspect per-epoch deltas.")